# 10. Machine learning II — inspeksi kualitas tablet sintetis

**Capaian:** Membuat fitur ukur, split antar-batch, model prediksi, dan error analysis.

**Prasyarat:** Python dasar, NumPy, dan membaca array. **Estimasi:** 75–100 menit.

> Kerjakan dari atas ke bawah. Setiap grafik wajib disertai penafsiran, bukan hanya ditempel.

## Konsep dan model

Studi kasus simulasi inspeksi tablet (*quality control*). Kelas 0 = utuh, 1 = cacat sintetis. Fitur: area, sirkularitas, soliditas, eksentrisitas, intensitas, variasi intensitas, tepi. **Penting:** label sintetis mengikuti aturan generator; hasil bagus tidak menyatakan akurasi di pabrik.

## Visualisasi dataset dan sebaran batch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pengolahan_citra_id.datasets import tablet_sintetis
from pengolahan_citra_id.features import fitur_tablet,FITUR_TABLET
from pengolahan_citra_id.evaluation import split_berdasar_batch,metrik_klasifikasi
images,y,batch=tablet_sintetis(n_batch=20,per_batch=12,seed=42)
fig,ax=plt.subplots(2,4,figsize=(10,5))
for a,i in zip(ax.ravel(),range(8)):
    a.imshow(images[i],cmap='gray',vmin=0,vmax=255);a.set_title(f'Label={y[i]}, batch={batch[i]}');a.axis('off')
plt.show()

## Feature engineering

In [ ]:
X=np.stack([fitur_tablet(image) for image in images])
print('Bentuk fitur:',X.shape)
for col,name in enumerate(FITUR_TABLET):
    print(f'{name:23s} kelas 0={X[y==0,col].mean():.3f} kelas 1={X[y==1,col].mean():.3f}')

## Pemisahan antarbatches dan model

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
tr,te=split_berdasar_batch(y,batch,test_size=0.25,seed=42)
assert not set(batch[tr]) & set(batch[te])
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42))
model.fit(X[tr],y[tr]);pred=model.predict(X[te])
print(metrik_klasifikasi(y[te],pred))
ConfusionMatrixDisplay.from_predictions(y[te],pred,display_labels=['Utuh','Cacat']);plt.show()

## Ablasi fitur dan baseline mayoritas

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
major=DummyClassifier(strategy='most_frequent').fit(X[tr],y[tr]); p_major=major.predict(X[te])
short=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42))
short.fit(X[tr,:4],y[tr]);p_short=short.predict(X[te,:4])
print('F1 mayoritas:',f1_score(y[te],p_major,average='macro'))
print('F1 fitur bentuk saja:',f1_score(y[te],p_short,average='macro'))
print('F1 semua fitur:',f1_score(y[te],pred,average='macro'))

## Diskusi, HOTS, dan latihan

1. Jika biaya *false negative* tinggi, bagaimana threshold klasifikasi perlu diubah? Gunakan probabilitas `predict_proba`.
2. Ciptakan kondisi gangguan (defocus, glare, variasi ukuran). Uji pada batch terpisah.
3. Tulis rencana validasi pada citra tablet nyata meliputi akuisisi, anotasi, ground truth, dan ketidakpastian.
4. Jelaskan mengapa 100% skor sintetis bisa merupakan indikator tugas terlalu mudah.

**Laporan singkat:** rumusan masalah → rancangan eksperimen → hasil → interpretasi → keterbatasan → kesimpulan.

## Catatan keselamatan dan validitas

Dataset ini **hanya simulasi** dengan pola cacat yang sengaja dibuat jelas. Model wajib divalidasi ulang terhadap pengukuran dan label manusia sebelum dipertimbangkan untuk inspeksi nyata.